In [ ]:
# Cell 1 - install/upgrade core libs (latest stable releases)
!pip install -q transformers datasets evaluate accelerate scikit-learn wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 53.4 MB/s eta 0:00:00


In [ ]:
# Cell 2 - mount drive and load dataset
from google.colab import drive
drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/Research/NLP_Multimodal_LLM/data/one_llm_query_logs_table.csv"

import pandas as pd
df = pd.read_csv(CSV_PATH)

Mounted at /content/drive


In [ ]:
# Cell 3 - Cleans dataframe to include only the input/output cols and setup dataset from the dataframe
LABEL_COL   = "likert_1"
DROP_COLS   = {"query_ID", "user_id", "llm_name_1", "query_timestamp"}
TEXT_COLS = [c for c in df.columns if (c not in DROP_COLS) and (c != LABEL_COL)]

SEP = " [SEP] "            # BERT‑friendly separator

# concat text columns into one new column of data
INPUT_COL = "input_text"
df[INPUT_COL] = df[TEXT_COLS].astype(str).agg(SEP.join, axis=1)
# drop all non-relevant columns(keep input_text - inputs and likert_1 - labels)
df = df[[INPUT_COL, LABEL_COL]]

# Normalize the ratings from 1 - 5 to 0 - 4(to match with the model)
# and make sure its a float for regression training
df[LABEL_COL] = (df[LABEL_COL] - 1).astype(float)

from datasets import Dataset
dataset = Dataset.from_pandas(df[["input_text", LABEL_COL]])

df

,input_text,likert_1
0,28 [SEP] Where is Romania? [SEP] - **Location*...,3.0
1,4 [SEP] what is single player [SEP] - **Single...,3.0
2,28 [SEP] What is the most famous food in Roman...,3.0
3,4 [SEP] How do I maximize time playing single ...,3.0
4,4 [SEP] what are the best single player tips [...,3.0
5,28 [SEP] Who are the most famous people from R...,4.0
6,4 [SEP] can single player turn into multiplaye...,3.0
7,2 [SEP] Who was Mark's closest friend in Macro...,0.0
8,2 [SEP] Could people on the severed floor reme...,4.0
9,2 [SEP] Why did Mark start working on the seve...,3.0


In [ ]:
# Cell 4 - Tokenize the Dataset
from transformers import AutoTokenizer

# Use a fast and effective model checkpoint
MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# Tokenization function
def tokenize(batch):
  return tokenizer(batch["input_text"], padding=True, truncation=True, max_length=512)

# Apply tokenization to the entire dataset
dataset_encoded = dataset.map(tokenize, batched=True, batch_size=None) # Setting batch_size=None processes the whole dataset at once

# Rename the label column to "labels" for the trainer
dataset_encoded = dataset_encoded.rename_columns({LABEL_COL: "labels"})

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

In [ ]:
# Cell 5 - Define Metrics and Model
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats

# Performance metrics for a regression task
def compute_metrics_for_regression(eval_pred):
    logits, labels = eval_pred

    # Calculate standard regression metrics
    mse = mean_squared_error(labels, logits)
    mae = mean_absolute_error(labels, logits)
    r2 = r2_score(labels, logits)
    pr = scipy.stats.pearsonr(labels.flatten(), logits.flatten())[0]
    sr = scipy.stats.spearmanr(labels.flatten(), logits.flatten())[0]

    # Calculate custom "accuracy" (predictions within 0.5 of the true label)
    errors = np.abs(logits.flatten() - labels.flatten())
    accuracy = np.sum(errors < 1) / len(errors)

    return {"mse": mse, "mae": mae, "r2": r2, "pearson_r": pr, "spearman_r": sr, "accuracy": accuracy}


# CRITICAL CHANGE: Set num_labels to 1 for regression
num_labels = 1
print(f"Configuring model for regression with {num_labels} output.")

# Load the model with a regression head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=num_labels)

Configuring model for regression with 1 output.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from google.colab import userdata
from transformers import TrainingArguments, Trainer
import gc
import os
import wandb # Import wandb

# --- 1. Setup ---
full_dataset_encoded = dataset_encoded
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')

# Give this entire CV experiment a single group name
WANDB_GROUP_NAME = f"CV-bert-base-{pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M')}"

all_fold_metrics = []

# --- 2. Cross-Validation Loop ---
for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset_encoded)):
    print(f"=============== FOLD {fold+1}/{N_SPLITS} ===============")

    # -- A. Create datasets --
    train_fold = full_dataset_encoded.select(train_idx)
    eval_fold = full_dataset_encoded.select(val_idx)

    # -- B. Re-initialize Model --
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=num_labels)

    # -- C. Explicitly initialize W&B for this fold --
    # The trainer will automatically use this active run
    wandb.init(
        project="one-llm-regression-keep-userquery-llmresponse1-adjustment",  # Specify your W&B project name
        group=WANDB_GROUP_NAME,
        name=f"fold-{fold+1}",
        reinit=True, # Important for loops in notebooks
    )

    # -- D. Define Training Arguments --
    output_dir = f"/content/drive/MyDrive/Research/NLP_Multimodal_LLM/output/{WANDB_GROUP_NAME}/fold-{fold+1}"

    # We no longer need run_name here as wandb.init() handles it
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="mse",
        greater_is_better=False,
        logging_strategy="epoch",
        load_best_model_at_end=True,
        push_to_hub=False,
        # To avoid cluttering the output, we can disable the progress bar for inner loops
    )

    # -- D. Initialize and run the Trainer for this fold --
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=eval_fold,
        compute_metrics=compute_metrics_for_regression,
        tokenizer=tokenizer
    )

    trainer.train()
    metrics = trainer.evaluate()
    all_fold_metrics.append(metrics)
    print(f"Metrics for Fold {fold+1}: {metrics}")

    wandb.finish() # End the W&B run for this fold

    # -- F. Clean up --
    del model
    del trainer
    gc.collect()


# --- 3. Aggregate and Display Final Results ---
print("\n=============== Cross-Validation Complete ===============")
print(f"Metrics were calculated over {N_SPLITS} folds.")

# Create a DataFrame to easily calculate mean and std dev
results_df = pd.DataFrame(all_fold_metrics)

# Display mean and standard deviation for each metric
print("\nAverage Metrics (Mean +/- Std Dev):")
for metric in results_df.columns:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std()
    print(f"- {metric}: {mean_val:.4f} +/- {std_val:.4f}")

=============== FOLD 1/5 ===============


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: Currently logged in as: aryan-sajith (aryan-sajith-university-of-massachusetts) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


<ipython-input-6-528137293>:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Mse,Mae,R2,Pearson R,Spearman R,Accuracy
1,7.442000,10.345819,10.345819,3.172794,-42.107578,-0.502697,-0.288675,0.000000
2,6.291700,9.202042,9.202041,2.986696,-37.341835,-0.405114,-0.288675,0.000000
3,5.227100,8.058278,8.058277,2.787591,-32.576153,-0.351648,-0.288675,0.000000
4,6.047100,6.961064,6.961065,2.585199,-28.004436,-0.239418,-0.288675,0.000000
5,4.680200,6.092193,6.092193,2.415569,-24.384138,-0.076103,-0.288675,0.000000
6,4.107700,5.411409,5.411409,2.274458,-21.547537,0.110173,0.288675,0.000000
7,2.704300,4.879530,4.879530,2.157769,-19.331375,0.285863,0.288675,0.000000
8,2.969000,4.501436,4.501435,2.070117,-17.755980,0.368526,0.288675,0.000000
9,3.248100,4.272264,4.272264,2.014662,-16.801096,0.388530,0.288675,0.000000
10,1.921900,4.176883,4.176883,1.990980,-16.403677,0.388925,0.288675,0.000000


Metrics for Fold 1: {'eval_loss': 4.176882743835449, 'eval_mse': 4.176882743835449, 'eval_mae': 1.9909803867340088, 'eval_r2': -16.403676986694336, 'eval_pearson_r': 0.3889249563217163, 'eval_spearman_r': 0.28867513459481287, 'eval_accuracy': 0.0, 'eval_runtime': 0.1326, 'eval_samples_per_second': 37.706, 'eval_steps_per_second': 7.541, 'epoch': 10.0}


...

<ipython-input-6-528137293>:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Mse,Mae,R2,Pearson R,Spearman R,Accuracy
1,8.788200,4.945760,4.945760,2.067193,-5.594347,0.733065,0.774597,0.250000
2,7.162800,4.060190,4.060190,1.852293,-4.413586,0.785154,0.774597,0.250000
3,6.100100,3.188884,3.188884,1.620849,-3.251845,0.893182,0.774597,0.250000
4,4.908500,2.437930,2.437930,1.394910,-2.250573,0.973167,0.774597,0.250000
5,3.744400,1.923363,1.923364,1.210311,-1.564485,0.996069,0.774597,0.250000
6,3.386600,1.565758,1.565758,1.103882,-1.087677,0.997789,0.774597,0.250000
7,2.968600,1.308932,1.308932,1.030571,-0.745242,0.996159,0.774597,0.250000
8,2.371700,1.129893,1.129893,0.971904,-0.506524,0.994850,0.774597,0.250000
9,2.243900,1.022298,1.022298,0.932783,-0.363063,0.993954,0.774597,0.250000
10,2.388700,0.978869,0.978869,0.916108,-0.305158,0.993594,0.774597,0.250000


Metrics for Fold 5: {'eval_loss': 0.978868842124939, 'eval_mse': 0.978868842124939, 'eval_mae': 0.9161080121994019, 'eval_r2': -0.30515849590301514, 'eval_pearson_r': 0.9935935735702515, 'eval_spearman_r': 0.7745966692414834, 'eval_accuracy': 0.25, 'eval_runtime': 0.1193, 'eval_samples_per_second': 33.533, 'eval_steps_per_second': 8.383, 'epoch': 10.0}


eval/accuracy,▁▁▁▁▁▁▁▁▁▁▁
eval/loss,█▆▅▄▃▂▂▁▁▁▁
eval/mae,█▇▅▄▃▂▂▁▁▁▁
eval/mse,█▆▅▄▃▂▂▁▁▁▁
eval/pearson_r,▁▂▅▇███████
eval/r2,▁▃▄▅▆▇▇████
eval/runtime,▁▁▁▂▂▂▁▂▇▂█
eval/samples_per_second,▇▇█▆▆▇█▆▁▇▁
eval/spearman_r,▁▁▁▁▁▁▁▁▁▁▁
eval/steps_per_second,▇▇█▆▆▇█▆▁▇▁
train/epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇████



=============== Cross-Validation Complete ===============
Metrics were calculated over 5 folds.

Average Metrics (Mean +/- Std Dev):
- eval_loss: 2.4411 +/- 1.1433
- eval_mse: 2.4411 +/- 1.1433
- eval_mae: 1.4501 +/- 0.3855
- eval_r2: -6.5319 +/- 8.2176
- eval_pearson_r: 0.6102 +/- 0.3525
- eval_spearman_r: 0.4985 +/- 0.3136
- eval_accuracy: 0.1300 +/- 0.1204
- eval_runtime: 0.1222 +/- 0.0063
- eval_samples_per_second: 39.3202 +/- 3.7916
- eval_steps_per_second: 8.1992 +/- 0.4086
- epoch: 10.0000 +/- 0.0000
